# 08 — Gold Layer: Business Metrics & SARIMA Demand Forecast

This notebook has two sections:

**Section A — Descriptive KPIs**: Aggregations over `fact_trips` + dimension joins.

**Section B — Predictive Model (SARIMA)**:
1. Build an hourly demand time series from `fact_trips`.
2. Apply the **Augmented Dickey-Fuller (ADF) test** to confirm non-stationarity.
3. Plot **ACF / PACF** to identify seasonal lags (s=24h, s=168h=7 days).
4. Fit a **SARIMA(p,d,q)(P,D,Q)[s]** model.
5. Generate a **7-day demand forecast** with confidence intervals.

> *Justification for the jury:*  
> "We evaluated the demand series and detected recurring seasonal patterns every
> 24 hours and every 7 days via ACF/PACF. The ADF test confirmed non-stationarity
> (p > 0.05), so we rejected classical ARIMA and chose **SARIMA** to capture the
> inherent seasonality of Manhattan taxi demand."

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
import pyspark.sql.functions as F

# Statistical / ML libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

print("Imports OK.")

In [ ]:
spark = get_spark("TLC_Gold_Metrics")

---
# Section A — Descriptive KPIs

In [ ]:
fact_trips  = read_mongo(spark, settings.MONGO_DB_GOLD, "fact_trips")
dim_zone    = read_mongo(spark, settings.MONGO_DB_GOLD, "dim_zone")
dim_vehicle = read_mongo(spark, settings.MONGO_DB_GOLD, "dim_vehicle")
dim_date    = read_mongo(spark, settings.MONGO_DB_GOLD, "dim_date")

print(f"fact_trips: {fact_trips.count():,} rows")

In [ ]:
# KPI 1 — Total Trips and Revenue by Vehicle Type
kpi1 = (
    fact_trips
    .groupBy("vehicle_id")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_distance_miles"),
        F.avg("duration_minutes").alias("avg_duration_min"),
    )
    .join(dim_vehicle, fact_trips["vehicle_id"] == dim_vehicle["id"], "left")
    .select("name", "total_trips", "total_revenue", "avg_distance_miles", "avg_duration_min")
    .orderBy(F.col("total_trips").desc())
)
print("KPI 1 — Trips & Revenue by Vehicle Type")
kpi1.show(truncate=False)

In [ ]:
# KPI 2 — Top 10 Pickup Zones by Revenue
kpi2 = (
    fact_trips
    .groupBy("pickup_zone_id")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.count("*").alias("total_trips"),
    )
    .join(dim_zone, fact_trips["pickup_zone_id"] == dim_zone["zone_id"], "left")
    .select("zone_name", "borough", "total_trips", "total_revenue")
    .orderBy(F.col("total_revenue").desc())
    .limit(10)
)
print("KPI 2 — Top 10 Pickup Zones by Revenue")
kpi2.show(truncate=False)

In [ ]:
# KPI 3 — Congestion Fee Impact by Vehicle Type
kpi3 = (
    fact_trips
    .groupBy("vehicle_id", "pickup_date_id")
    .agg(F.sum("cbd_congestion_fee").alias("total_congestion_fee"))
    .orderBy("pickup_date_id", "vehicle_id")
)
print("KPI 3 — Daily Congestion Fee by Vehicle Type")
kpi3.show(20, truncate=False)

In [ ]:
# KPI 4 — Hourly demand profile (for SARIMA input)
kpi4 = (
    fact_trips
    .groupBy("pickup_hour")
    .agg(F.count("*").alias("trips"))
    .orderBy("pickup_hour")
)
print("KPI 4 — Total Trips by Hour of Day")
kpi4.show(24, truncate=False)

---
# Section B — SARIMA Demand Forecast

## B1. Build Hourly Demand Time Series

We aggregate trip counts by **hour** from the Gold fact table.  
Yellow taxi is used as the primary signal (highest volume, most complete financials).

In [ ]:
# Build hourly time series from fact_trips
# pickup_date_id is yyyyMMdd integer, pickup_hour is 0-23
hourly_spark = (
    fact_trips
    .filter(F.col("vehicle_id") == "yellow")  # focus on yellow taxi
    .withColumn(
        "pickup_dt",
        F.to_timestamp(
            F.concat(
                F.col("pickup_date_id").cast("string"),
                F.lpad(F.col("pickup_hour").cast("string"), 2, "0"),
            ),
            "yyyyMMddHH",
        ),
    )
    .groupBy("pickup_dt")
    .agg(F.count("*").alias("demand"))
    .orderBy("pickup_dt")
)

# Collect to Pandas for statistical modeling
ts_pd = hourly_spark.toPandas()
ts_pd.set_index("pickup_dt", inplace=True)
ts_pd.index = pd.to_datetime(ts_pd.index)
ts_pd = ts_pd.sort_index()
ts_pd = ts_pd.asfreq("H", fill_value=0)  # ensure complete hourly index

print(f"Time series: {len(ts_pd)} hourly observations")
print(f"Date range: {ts_pd.index.min()} → {ts_pd.index.max()}")
print(ts_pd.describe())

In [ ]:
# Plot raw demand series
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(ts_pd.index, ts_pd["demand"], linewidth=0.5, alpha=0.8, color="#3b82f6")
ax.set_title("Hourly Yellow Taxi Demand (Full Series)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Trip Count")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.savefig("/home/jovyan/work/data/sarima_01_raw_series.png", dpi=120)
plt.show()
print("Chart saved.")

## B2. Augmented Dickey-Fuller (ADF) Test

**Hypothesis:**
- H₀: The series has a unit root (non-stationary)
- H₁: The series is stationary

If p-value > 0.05 → fail to reject H₀ → series is **non-stationary** → differencing needed → SARIMA, not ARIMA.

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(ts_pd["demand"].dropna(), autolag="AIC")

adf_stat  = adf_result[0]
p_value   = adf_result[1]
n_lags    = adf_result[2]
crit_vals = adf_result[4]

print("══════════════════════════════════════════")
print("  Augmented Dickey-Fuller Test Results")
print("══════════════════════════════════════════")
print(f"  ADF Statistic : {adf_stat:.4f}")
print(f"  p-value       : {p_value:.6f}")
print(f"  Lags Used     : {n_lags}")
for key, val in crit_vals.items():
    print(f"  Critical Value ({key}): {val:.4f}")

print()
if p_value > 0.05:
    print("✗ RESULT: Fail to reject H₀ — series is NON-STATIONARY")
    print("  → Differencing required (d ≥ 1)")
    print("  → Seasonal differencing also expected (D=1, s=24 or s=168)")
    print("  → ARIMA is insufficient. SARIMA selected.")
else:
    print("✓ RESULT: Reject H₀ — series is STATIONARY (d=0)")
    print("  → SARIMA still preferred if ACF/PACF show seasonal spikes.")

## B3. ACF / PACF Analysis

Significant spikes at lags **24** and **168** confirm seasonality:
- **s=24**: intra-day pattern (morning/evening rush)
- **s=168**: weekly pattern (weekday vs. weekend demand)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Use first-differenced series for cleaner ACF/PACF
ts_diff = ts_pd["demand"].diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

plot_acf(
    ts_diff,
    lags=200,
    ax=axes[0],
    color="#3b82f6",
    title="ACF — First-Differenced Hourly Demand (lags=200)",
)
axes[0].axvline(x=24,  color="red",    linestyle="--", alpha=0.7, label="lag=24h")
axes[0].axvline(x=168, color="orange", linestyle="--", alpha=0.7, label="lag=168h (7 days)")
axes[0].legend()

plot_pacf(
    ts_diff,
    lags=50,
    ax=axes[1],
    method="ywm",
    color="#10b981",
    title="PACF — First-Differenced Hourly Demand (lags=50)",
)
axes[1].axvline(x=24, color="red", linestyle="--", alpha=0.7, label="lag=24h")
axes[1].legend()

plt.tight_layout()
plt.savefig("/home/jovyan/work/data/sarima_02_acf_pacf.png", dpi=120)
plt.show()
print("ACF/PACF chart saved.")

## B4. Fit SARIMA Model

Based on ADF and ACF/PACF analysis:
- **Non-seasonal**: `(p=1, d=1, q=1)` — AR(1) after first differencing
- **Seasonal**: `(P=1, D=1, Q=1)[s=24]` — daily seasonality with seasonal differencing

For the exam demo we use `s=24` (hourly). If data is aggregated daily, switch to `s=7`.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Use last 30 days of data for faster training during demo
# Change to ts_pd["demand"] for full training
TRAIN_DAYS = 30
train_series = ts_pd["demand"].iloc[-(TRAIN_DAYS * 24):]

print(f"Training on {len(train_series):,} hourly observations ({TRAIN_DAYS} days)")
print(f"Period: {train_series.index[0]} → {train_series.index[-1]}")

# SARIMA(1,1,1)(1,1,1)[24]
sarima_order         = (1, 1, 1)
sarima_seasonal_order = (1, 1, 1, 24)   # s=24 hours

print(f"\nFitting SARIMA{sarima_order}×{sarima_seasonal_order}...")

model = SARIMAX(
    train_series,
    order=sarima_order,
    seasonal_order=sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = model.fit(disp=False)

print("\n── Model Summary (key metrics) ──")
print(f"  AIC  : {sarima_fit.aic:.2f}")
print(f"  BIC  : {sarima_fit.bic:.2f}")
print(f"  HQIC : {sarima_fit.hqic:.2f}")

In [ ]:
# Full model summary (for the exam)
print(sarima_fit.summary())

## B5. 7-Day Demand Forecast

In [ ]:
FORECAST_HOURS = 7 * 24  # 7 days

forecast = sarima_fit.get_forecast(steps=FORECAST_HOURS)
forecast_mean = forecast.predicted_mean
conf_int      = forecast.conf_int(alpha=0.05)  # 95% CI

print(f"Forecast generated: {FORECAST_HOURS} hours ({FORECAST_HOURS // 24} days)")
print(f"Forecast range: {forecast_mean.index[0]} → {forecast_mean.index[-1]}")
print(f"Average predicted demand: {forecast_mean.mean():.0f} trips/hour")

In [ ]:
# Forecast visualization
fig, ax = plt.subplots(figsize=(16, 5))

# Historical: last 7 days of training data
history = train_series.iloc[-(7 * 24):]
ax.plot(
    history.index, history.values,
    label="Historical Demand (last 7 days)",
    color="#3b82f6", linewidth=1.2
)

# Forecast
ax.plot(
    forecast_mean.index, forecast_mean.values,
    label="SARIMA Forecast (7 days)",
    color="#f59e0b", linewidth=1.5, linestyle="--"
)

# 95% Confidence interval
ax.fill_between(
    forecast_mean.index,
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    color="#f59e0b", alpha=0.15,
    label="95% Confidence Interval"
)

# Vertical divider at forecast start
ax.axvline(
    x=forecast_mean.index[0],
    color="red", linestyle=":", alpha=0.7, label="Forecast Start"
)

ax.set_title(
    f"SARIMA{sarima_order}×{sarima_seasonal_order} — Yellow Taxi Hourly Demand Forecast",
    fontsize=13, fontweight="bold"
)
ax.set_xlabel("Date")
ax.set_ylabel("Trips / Hour")
ax.legend(loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/home/jovyan/work/data/sarima_03_forecast.png", dpi=150)
plt.show()
print("Forecast chart saved.")

## B6. Persist Forecast to Gold Layer

In [ ]:
# Convert forecast to Spark DataFrame and store in tlc_gold.demand_forecast
forecast_df = pd.DataFrame({
    "forecast_dt":   forecast_mean.index,
    "predicted_demand": forecast_mean.values.round(0).astype(int),
    "ci_lower":      conf_int.iloc[:, 0].values.round(0).astype(int),
    "ci_upper":      conf_int.iloc[:, 1].values.round(0).astype(int),
    "model":         f"SARIMA{sarima_order}×{sarima_seasonal_order}",
    "vehicle_type":  "yellow",
    "generated_at":  pd.Timestamp.now(),
})

forecast_spark = spark.createDataFrame(forecast_df)
write_mongo(forecast_spark, settings.MONGO_DB_GOLD, "demand_forecast", mode="overwrite")
print(f"Forecast saved to tlc_gold.demand_forecast: {len(forecast_df)} rows")
forecast_df.head(10)